# CogVLM 入门教程

CogVLM 是清华大学提出的视觉语言模型，通过 **Visual Expert** 模块实现深度视觉-语言融合。

## 学习目标

- 理解 Visual Expert 的设计思想
- 掌握 CogVLM 的架构特点
- 学会使用 CogVLM 进行多模态理解

## 目录

1. [什么是 CogVLM](#1-什么是-cogvlm)
2. [Visual Expert 模块](#2-visual-expert-模块)
3. [模型架构](#3-模型架构)
4. [代码实践](#4-代码实践)
5. [总结](#5-总结)

## 1. 什么是 CogVLM

### 背景问题

传统视觉语言模型（如 LLaVA）使用简单的线性投影连接视觉和语言：
- 视觉特征被"压缩"到语言空间
- 可能丢失重要的视觉信息

### CogVLM 的解决方案

CogVLM 引入 **Visual Expert** 模块：
- 在每个 Transformer 层添加专门处理视觉 token 的参数
- 视觉和文本 token 使用不同的 QKV 投影和 FFN
- 保留更多视觉细节，提升多模态理解能力

In [ ]:
# 环境准备
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

torch.manual_seed(42)

## 2. Visual Expert 模块

### 核心思想

```
普通 LLM:     所有 token → 共享 QKV/FFN
CogVLM:       视觉 token → Visual Expert QKV/FFN
              文本 token → 原始 LLM QKV/FFN
```

### 优势

1. **保留 LLM 能力**: 文本处理使用原始参数
2. **增强视觉理解**: 视觉 token 有专门的处理路径
3. **深度融合**: 每一层都进行视觉-语言交互

In [ ]:
# 可视化 Visual Expert 概念
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 传统方法
ax = axes[0]
ax.set_title('传统方法 (LLaVA)', fontsize=12, fontweight='bold')
ax.add_patch(plt.Rectangle((0.1, 0.6), 0.35, 0.25, color='lightblue', ec='black'))
ax.text(0.275, 0.725, '视觉 Token', ha='center', fontsize=10)
ax.add_patch(plt.Rectangle((0.55, 0.6), 0.35, 0.25, color='lightgreen', ec='black'))
ax.text(0.725, 0.725, '文本 Token', ha='center', fontsize=10)
ax.annotate('', xy=(0.5, 0.4), xytext=(0.275, 0.6), arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('', xy=(0.5, 0.4), xytext=(0.725, 0.6), arrowprops=dict(arrowstyle='->', color='gray'))
ax.add_patch(plt.Rectangle((0.3, 0.2), 0.4, 0.2, color='lightyellow', ec='black'))
ax.text(0.5, 0.3, '共享 QKV/FFN', ha='center', fontsize=10)
ax.axis('off')

# CogVLM
ax = axes[1]
ax.set_title('CogVLM (Visual Expert)', fontsize=12, fontweight='bold')
ax.add_patch(plt.Rectangle((0.1, 0.6), 0.35, 0.25, color='lightblue', ec='black'))
ax.text(0.275, 0.725, '视觉 Token', ha='center', fontsize=10)
ax.add_patch(plt.Rectangle((0.55, 0.6), 0.35, 0.25, color='lightgreen', ec='black'))
ax.text(0.725, 0.725, '文本 Token', ha='center', fontsize=10)
ax.annotate('', xy=(0.2, 0.4), xytext=(0.275, 0.6), arrowprops=dict(arrowstyle='->', color='blue'))
ax.annotate('', xy=(0.8, 0.4), xytext=(0.725, 0.6), arrowprops=dict(arrowstyle='->', color='green'))
ax.add_patch(plt.Rectangle((0.05, 0.2), 0.35, 0.2, color='lightblue', ec='black', alpha=0.5))
ax.text(0.225, 0.3, 'Visual Expert', ha='center', fontsize=9)
ax.add_patch(plt.Rectangle((0.6, 0.2), 0.35, 0.2, color='lightgreen', ec='black', alpha=0.5))
ax.text(0.775, 0.3, 'LLM 原始参数', ha='center', fontsize=9)
ax.axis('off')

plt.tight_layout()
plt.show()

## 3. 模型架构

CogVLM 的完整架构：

1. **视觉编码器**: EVA-CLIP ViT
2. **MLP 适配器**: 将视觉特征投影到语言空间
3. **语言模型**: 带 Visual Expert 的 LLM
4. **位置编码**: RoPE (旋转位置编码)

In [ ]:
from cogvlm import CogVLMConfig, CogVLM

# 创建小型模型用于演示
config = CogVLMConfig(
    image_size=224,
    patch_size=14,
    vision_layers=4,
    vision_width=256,
    vision_heads=4,
    hidden_size=512,
    num_layers=4,
    num_heads=8,
    num_kv_heads=8,
    intermediate_size=1024,
    visual_expert_intermediate_size=1024,
    vocab_size=32000
)

model = CogVLM(config).to(device)
print(f'模型参数量: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 查看模型结构
print('=== CogVLM 模型组件 ===')
print(f'视觉编码器层数: {config.vision_layers}')
print(f'语言模型层数: {config.num_layers}')
print(f'隐藏维度: {config.hidden_size}')
print(f'注意力头数: {config.num_heads}')
print(f'KV 头数 (GQA): {config.num_kv_heads}')

## 4. 代码实践

### 4.1 提取视觉特征

In [ ]:
# 模拟图像输入
images = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    vision_features = model.get_vision_features(images)

print(f'视觉特征形状: {vision_features.shape}')
print(f'每张图像的 token 数: {vision_features.shape[1]}')

### 4.2 多模态前向传播

In [ ]:
# 模拟输入
batch_size = 2
seq_len = 300  # 需要足够长以容纳视觉 token

input_ids = torch.randint(0, 32000, (batch_size, seq_len)).to(device)
images = torch.randn(batch_size, 3, 224, 224).to(device)

with torch.no_grad():
    outputs = model(input_ids, images)

print(f'输出 logits 形状: {outputs["logits"].shape}')

### 4.3 Visual Expert 的作用

In [ ]:
# 统计 Visual Expert 参数量
total_params = sum(p.numel() for p in model.parameters())
vision_params = sum(p.numel() for p in model.vision_encoder.parameters())
expert_params = sum(
    p.numel() for name, p in model.named_parameters() 
    if 'visual_expert' in name or 'vision_' in name
)

print(f'总参数: {total_params/1e6:.1f}M')
print(f'视觉编码器: {vision_params/1e6:.1f}M')
print(f'Visual Expert 相关: {expert_params/1e6:.1f}M')

## 5. 总结

### CogVLM 的核心创新

| 组件 | 作用 |
|------|------|
| Visual Expert Attention | 视觉 token 专用 QKV 投影 |
| Visual Expert FFN | 视觉 token 专用前馈网络 |
| EVA-CLIP 编码器 | 高质量视觉特征提取 |
| RoPE | 旋转位置编码 |

### 与其他模型对比

| 模型 | 视觉-语言融合方式 |
|------|------------------|
| LLaVA | 线性投影 |
| BLIP-2 | Q-Former |
| CogVLM | Visual Expert |

### 适用场景

- 图像理解和描述
- 视觉问答 (VQA)
- 多轮视觉对话